In [1]:
import sys
sys.path.append('../..')
import numpy as np
from cpm.generators.wrapper import MetaSignalDetectionWrapper
from cpm.models.signal_detection import MetaSignalDetectionHelper, MetaSignalDetectionModel
from cpm.optimisation import minimise, DifferentialEvolution, FminBound, Bads
from cpm.applications.MetacognitionSDT import metacognitionSDT_initparams

In [2]:
nR_S1 = [36, 24, 17, 20, 10, 12, 9, 2]
nR_S2 = [1, 4, 10, 11, 19, 18, 28, 39]
nbins = 4

binned_data = {
    'nR_S1': np.array(nR_S1),
    'nR_S2': np.array(nR_S2),
}

In [3]:
params, d1, c1, data_pandas = metacognitionSDT_initparams(binned_data=binned_data, nbins=4)
data_pandas["ppt"] = 0

print(params.t2c1.value)

[[20 17 24 36]
 [10 12  9  2]
 [19 18 28 39]
 [11 10  4  1]]
   nbins                                           observed
0      4  [[20, 17, 24, 36], [10, 12, 9, 2], [19, 18, 28...
[ 0.14418615  0.03484482 -0.0421968  -0.03260732 -0.10486451  0.1026477 ]


In [4]:
def model(parameters):

    parameters.t2c1.value = np.sort(parameters.t2c1.value)
    msd = MetaSignalDetectionModel(nbins=nbins, d1=parameters.d1, t1c1=parameters.t1c1, meta_d1=parameters.meta_d1, t2c1=parameters.t2c1)
    output = {
        "nbins": nbins,
        "dependent": msd.t2_probs(),
    }
    return output

print(model(params))

1.2395747659040524
[ 0.14418615  0.03484482 -0.0421968  -0.03260732 -0.10486451  0.1026477 ]
[-0.10486451 -0.0421968  -0.03260732  0.03484482  0.1026477   0.14418615]
{'nbins': 4, 'dependent': array([[9.93457735e-01, 4.06871035e-03, 5.80288015e-04, 1.89326699e-03],
       [9.44630761e-01, 3.33946200e-02, 5.03159694e-03, 1.69430222e-02],
       [3.91026203e-02, 7.40367154e-02, 4.39369534e-02, 8.42923711e-01],
       [7.96240069e-02, 1.39638321e-01, 7.62916582e-02, 7.04446014e-01]])}


In [5]:
wrapper = MetaSignalDetectionWrapper(model=model, data=binned_data, parameters=params)

In [6]:
fit = Bads(
    model=wrapper,  # Wrapper class with the model we specified from before
    data=data_pandas.groupby("ppt"),
    minimisation=minimise.LogLikelihood.multinomial,
    parallel=False,
    prior=False,
    ppt_identifier="ppt",
    number_of_starts=5,
)

fit.optimise()

Starting optimization 1/5 from [  0.17543712 -23.26642182 -20.60803866 -38.58496291  28.83013193
  42.42838912  31.40668395]
bads:TooCloseBounds: For each variable, hard and plausible bounds should not be too close. Moving plausible bounds.
-20.615527343749996
[-23.24697266 -20.61552734 -38.59707031  28.85097656  42.44677734
  31.38496094]
[-38.59707031 -23.24697266 -20.61552734  28.85097656  31.38496094
  42.44677734]
-20.615527343749996
[-23.24697266 -20.61552734 -38.59707031  28.85097656  42.44677734
  31.38496094]
[-38.59707031 -23.24697266 -20.61552734  28.85097656  31.38496094
  42.44677734]
Beginning optimization of a DETERMINISTIC objective function

 Iteration    f-count         f(x)           MeshScale          Method             Actions
     0           2           1e+10               1                                 Uncertainty test
-96.0015625
[-50.19492187 -96.0015625  -35.67324219   5.11923828  52.48525391
  35.03974609]
[-96.0015625  -50.19492187 -35.67324219   5.11923

/home/mavoeh/projects/modelling-toolbox/.venv/lib64/python3.12/site-packages/gpyreg/covariance_functions.py:394: RuntimeWarning: divide by zero encountered in log
  lower_bounds[D] = np.log(height) + np.log(tol)
/home/mavoeh/projects/modelling-toolbox/.venv/lib64/python3.12/site-packages/gpyreg/covariance_functions.py:395: RuntimeWarning: divide by zero encountered in log
  upper_bounds[D] = np.log(height * 10)
/home/mavoeh/projects/modelling-toolbox/.venv/lib64/python3.12/site-packages/gpyreg/covariance_functions.py:396: RuntimeWarning: divide by zero encountered in log
  plausible_lower_bounds[D] = np.log(height) + 0.5 * np.log(tol)
/home/mavoeh/projects/modelling-toolbox/.venv/lib64/python3.12/site-packages/gpyreg/covariance_functions.py:397: RuntimeWarning: divide by zero encountered in log
  plausible_upper_bounds[D] = np.log(height)
/home/mavoeh/projects/modelling-toolbox/.venv/lib64/python3.12/site-packages/gpyreg/covariance_functions.py:398: RuntimeWarning: divide by zero encou

AttributeError: 'float' object has no attribute 'item'

In [ ]:
from scipy.stats import multinomial

n = np.array([[1,2,3,4],[1,2,3,4]])
p = np.array([[0.2,0.3,0.1,0.4],[0.1,0.2,0.3,0.4]])

print(n.shape, p.shape)

np.sum([multinomial.logpmf(n[i], 10.0, p[i]) for i in range(p.shape[0])])